In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [ ]:
import pandas as pd

# 1. Read the CSV files into pandas DataFrames    
credits = pd.read_csv("D:/Projects/Movie_Recomendation/data/tmdb_5000_credits.csv")
movies= pd.read_csv("D:/Projects/Movie_Recomendation/data/tmdb_5000_movies.csv")

credits.head(2)


In [ ]:
movies.shape
credits.shape

In [ ]:
movies= pd.merge(credits, movies, on="title", how="inner")

# 3. View the combined dataset
movies.head(1)

In [ ]:
# 1. Shape of the dataset (rows, columns)
print("Dataset Shape:", movies.shape)
print("-" * 40)

# 2. List all columns
print("All Columns:", movies.columns.tolist())
print("-" * 40)

# 3. Count missing values in each column
print("Missing Values per Column:\n", movies.isnull().sum())
print("-" * 40)




In [ ]:
# 4. Count total duplicate rows
print("Total Duplicate Rows:", movies.duplicated().sum())
print("-" * 40)

#5 display information about the dataset
print("Dataset Info:", movies.info())

In [ ]:
#print the summary statistics of the dataset
print(movies.describe(), "\n")
print("-" * 40)
print(movies.describe(include='object'), "\n")




In [ ]:
movies.dropna(subset=['overview'], inplace=True)

In [ ]:
movies = movies[
    [
        'movie_id',
        'title',
        'overview',
        'genres',
        'keywords',
        'cast',
        'crew'
    ]
]

movies.head(1)

In [ ]:
movies.iloc[0]

In [ ]:
def convert(text):
    L = []

    for item in ast.literal_eval(text):
        L.append(item['name'])

    return L

In [ ]:
movies['genres']=movies['genres'].apply(convert)

In [ ]:
movies['keywords'] = movies['keywords'].apply(convert)

In [ ]:
movies['genres'].head(1)

In [ ]:
movies['keywords'].head()

In [ ]:
import ast

def convert3(text):
    L = []

    counter = 0

    for item in ast.literal_eval(text):

        if counter != 3:
            L.append(item['name'])
            counter += 1

        else:
            break

    return L

In [ ]:
movies['cast'] = movies['cast'].apply(convert3)

In [ ]:
movies['cast'].head()

In [ ]:
def fetch_director(text):

    L = []

    for item in ast.literal_eval(text):

        if item['job'] == 'Director':
            L.append(item['name'])
            break

    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies['crew'].head()

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x: x.split())

In [ ]:
def collapse(L):
    return [i.replace(" ", "") for i in L]

In [ ]:
movies['genres'] = movies['genres'].apply(collapse)

movies['keywords'] = movies['keywords'].apply(collapse)

movies['cast'] = movies['cast'].apply(collapse)

movies['crew'] = movies['crew'].apply(collapse)

In [ ]:
movies['tags'] = (
    movies['overview']
    + movies['genres']
    + movies['keywords']
    + movies['cast']
    + movies['crew']
)

In [ ]:
new_df = movies[['movie_id', 'title', 'tags']]

In [ ]:
new_df.head()

In [130]:
#Convert the List into a Single String
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x))

In [131]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [133]:
def stem(text):
    y = []

    for word in text.split():
        y.append(ps.stem(word))

    return " ".join(y)

In [134]:
new_df['tags'] = new_df['tags'].apply(stem)

In [135]:
new_df['tags'] = new_df['tags'].apply(lambda x: x.lower())

In [138]:
from sklearn.feature_extraction.text import CountVectorizer


In [ ]:
cv = CountVectorizer(max_features=5000,stop_words='english')

In [140]:
vectors = cv.fit_transform(new_df['tags'])

In [142]:
vectors = vectors.toarray()

In [145]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vectors)

In [158]:
def recommend(movie):
    # Find the movie index
    movie_index = new_df[new_df['title'] == movie].index[0]

    # Get similarity scores
    distances = similarity[movie_index]

    # Sort by similarity (highest first)
    movies_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    # Print recommended movie titles
    for movie in movies_list:
        print(new_df.iloc[movie[0]].title)

In [157]:
recommend("Batman Begins")

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


In [151]:
import pickle
pickle.dump(new_df, open('movies.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))